# Questionário final dos profissionais

Comparação entre o G1 (sem material de apoio) e o G2 (com material de apoio): medianas, delta de Cliff, Mann-Whitney, Fisher, Spearman e correção de Holm.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

DADOS = Path("../dados")

# Verifica se as planilhas estão na pasta dados/ (ao lado da pasta notebooks/)
if not DADOS.exists():
    raise FileNotFoundError(f"Pasta de dados não encontrada: {DADOS.resolve()}. Crie a pasta dados/ ao lado de notebooks/ e coloque nela as planilhas.")


## Funções auxiliares

In [2]:
def coluna(df, trecho):
    """Encontra a coluna cujo título contém o trecho informado."""
    achadas = [c for c in df.columns if trecho.lower() in c.lower()]
    if not achadas:
        raise KeyError(f"Coluna não encontrada: {trecho}")
    return df[achadas[0]]


def delta_cliff(a, b):
    """Delta de Cliff: positivo quando b (G2) tende a ser maior que a (G1)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    maior = sum((y > x) for x in a for y in b)
    menor = sum((y < x) for x in a for y in b)
    return (maior - menor) / (len(a) * len(b))


def iqr(s):
    return s.quantile(0.75) - s.quantile(0.25)


def comparar(nome, a, b):
    """Mediana, média, intervalo interquartil, Mann-Whitney e delta de Cliff."""
    a, b = pd.Series(a, dtype=float), pd.Series(b, dtype=float)
    p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    return {"Variável": nome,
            "Md G1": round(a.median(), 2), "IIQ G1": round(iqr(a), 2),
            "Md G2": round(b.median(), 2), "IIQ G2": round(iqr(b), 2),
            "M G1": round(a.mean(), 2), "M G2": round(b.mean(), 2),
            "delta de Cliff": round(delta_cliff(a, b), 2), "p": round(p, 3)}


def holm(p):
    """Correção de Holm para comparações múltiplas."""
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); mx = 0
    for k, i in enumerate(o):
        mx = max(mx, min(1, (m - k) * p[i])); adj[i] = mx
    return adj

In [3]:
g1 = pd.read_excel(DADOS / "Questionario_Final_Grupo_1_respostas.xlsx")
g2 = pd.read_excel(DADOS / "Questionario_Final_Grupo_2_respostas.xlsx")


def contar_opcoes(serie, opcoes):
    """Conta quantas das opções listadas cada participante marcou."""
    return serie.fillna("").apply(lambda t: sum(op.lower() in t.lower() for op in opcoes))


def tem_outro(serie, opcoes, ignorar=()):
    """1 se o participante escreveu algo no campo "Outro" (texto fora das opções)."""
    def verificar(t):
        resto = re.sub(r"\([^)]*\)", "", t)
        for op in list(opcoes) + list(ignorar):
            resto = re.sub(re.escape(op) + r".*?(?=, [A-ZÁÉÍÓÚÂÊÔÃÕÇ]|$)", "", resto)
        return int(bool(re.sub(r"[\s,.;]", "", resto)))
    return serie.fillna("").apply(verificar)


escala_acess = {"Não levei": 1, "pouco": 2, "moderadamente": 3, "bastante": 4, "totalmente": 5}

def acessibilidade_considerada(df):
    serie = coluna(df, "acessibilidade em consideração").fillna("")
    return serie.apply(lambda t: next(v for k, v in escala_acess.items() if k.lower() in t.lower()))

recursos = ["Contraste de cores", "Textos alternativos", "Rótulos nos campos", "Hierarquia de títulos",
            "Navegação por teclado", "leitores de tela", "Tamanho de fonte", "Mensagens de erro claras"]
testes = ["Teste exploratório", "Testes unitários", "Testes E2E", "Teste de usabilidade",
          "Avaliação heurística", "Avaliação com IA", "Teste de acessibilidade",
          "Teste de responsividade", "Teste de compatibilidade", "Teste de desempenho", "Validação de código"]
problemas = ["Problemas de acessibilidade", "Não atendeu ao briefing", "Inventou informações",
             "Textos genéricos", "Imagens diferentes", "Código que não funcionou", "Código difícil",
             "Falhas de segurança", "Layout quebrado", "Problemas de desempenho", "Afirmou ter seguido regras"]

def variaveis(df):
    return pd.DataFrame({
        "briefing": coluna(df, "atendeu corretamente").astype(float),
        "corrigido": coluna(df, "precisou corrigir").astype(float),
        "revisao": coluna(df, "Com que frequência você revisou").astype(float),
        "esperado": coluna(df, "ficou como você esperava").astype(float),
        "acess": acessibilidade_considerada(df),
        "recursos": contar_opcoes(coluna(df, "incluiu de acessibilidade"), recursos),
        "testes": contar_opcoes(coluna(df, "tipos de teste"), testes),
        "problemas": contar_opcoes(coluna(df, "problemas você encontrou"), problemas)
                     + tem_outro(coluna(df, "problemas você encontrou"), problemas,
                                 ignorar=["Não encontrei problemas"]),
        "dificuldade": ~coluna(df, "principais dificuldades").fillna("")
                        .str.contains("Não tive dificuldades", case=False),
    })

v1, v2 = variaveis(g1), variaveis(g2)
print(f"G1: {len(v1)} participantes | G2: {len(v2)} participantes")

G1: 8 participantes | G2: 8 participantes


### 1.1 Comparação entre os grupos (Tabela 2)

In [4]:
tabela = pd.DataFrame([
    comparar("Atendimento ao briefing (1 a 5)", v1.briefing, v2.briefing),
    comparar("Conteúdo corrigido (1 a 5)", v1.corrigido, v2.corrigido),
    comparar("Frequência de revisão (1 a 5)", v1.revisao, v2.revisao),
    comparar("Resultado esperado (1 a 5)", v1.esperado, v2.esperado),
    comparar("Acessibilidade considerada (1 a 5)", v1.acess, v2.acess),
    comparar("Recursos de acessibilidade (0 a 8)", v1.recursos, v2.recursos),
    comparar("Tipos de teste realizados", v1.testes, v2.testes),
])
tabela

,Variável,Md G1,IIQ G1,Md G2,IIQ G2,M G1,M G2,delta de Cliff,p
0,Atendimento ao briefing (1 a 5),5.0,0.25,4.0,0.00,4.75,4.12,-0.62,0.017
1,Conteúdo corrigido (1 a 5),2.0,1.00,2.0,1.25,1.62,2.12,0.36,0.208
2,Frequência de revisão (1 a 5),3.5,1.00,4.0,1.25,3.50,3.88,0.22,0.470
3,Resultado esperado (1 a 5),4.0,0.50,4.0,1.00,4.00,3.62,-0.22,0.461
4,Acessibilidade considerada (1 a 5),3.0,0.50,4.0,1.00,2.88,3.62,0.42,0.149
5,Recursos de acessibilidade (0 a 8),4.5,3.25,7.0,2.75,4.50,6.00,0.31,0.310
6,Tipos de teste realizados,3.5,3.00,5.5,1.50,4.25,4.75,0.23,0.457


### 1.2 Testes exatos de Fisher

In [5]:
def fisher(nome, sim1, sim2, n1=len(v1), n2=len(v2)):
    p = stats.fisher_exact([[sim1, n1 - sim1], [sim2, n2 - sim2]]).pvalue
    print(f"{nome}: G1 {sim1}/{n1}, G2 {sim2}/{n2}, Fisher p = {p:.3f}")
    return p

p_dif = fisher("Relatou alguma dificuldade", int(v1.dificuldade.sum()), int(v2.dificuldade.sum()))
p_max = fisher("Nota máxima no briefing", int((v1.briefing == 5).sum()), int((v2.briefing == 5).sum()))

Relatou alguma dificuldade: G1 0/8, G2 7/8, Fisher p = 0.001
Nota máxima no briefing: G1 6/8, G2 1/8, Fisher p = 0.041


### 1.3 Correlações de Spearman (grupos reunidos)

In [6]:
todos = pd.concat([v1, v2], ignore_index=True)

def spearman(nome, x, y):
    r, p = stats.spearmanr(todos[x], todos[y])
    print(f"{nome}: rho = {r:.2f}; p = {p:.3f}")

spearman("Acessibilidade considerada x recursos", "acess", "recursos")
spearman("Revisão x tipos de teste", "revisao", "testes")
spearman("Tipos de teste x problemas encontrados", "testes", "problemas")
spearman("Briefing x tipos de teste", "briefing", "testes")
spearman("Briefing x recursos", "briefing", "recursos")

dif_mat = coluna(g2, "dificuldade para entender e aplicar").astype(float)
tempo = coluna(g2, "afetou o tempo").astype(float)
r, p = stats.spearmanr(dif_mat, tempo)
print(f"G2, dificuldade do material x tempo: rho = {r:.2f}; p = {p:.3f}")

Acessibilidade considerada x recursos: rho = 0.59; p = 0.017
Revisão x tipos de teste: rho = 0.54; p = 0.030
Tipos de teste x problemas encontrados: rho = 0.44; p = 0.087
Briefing x tipos de teste: rho = -0.28; p = 0.299
Briefing x recursos: rho = -0.04; p = 0.878
G2, dificuldade do material x tempo: rho = 0.69; p = 0.060


### 1.4 Correção de Holm

In [7]:
pvals = list(tabela["p"]) + [p_dif]
nomes = list(tabela["Variável"]) + ["Relatou alguma dificuldade"]
for n, p, pa in zip(nomes, pvals, holm(pvals)):
    print(f"{n}: p = {p:.3f} -> p ajustado = {pa:.3f} {'(significativo)' if pa < 0.05 else ''}")

Atendimento ao briefing (1 a 5): p = 0.017 -> p ajustado = 0.119 
Conteúdo corrigido (1 a 5): p = 0.208 -> p ajustado = 1.000 
Frequência de revisão (1 a 5): p = 0.470 -> p ajustado = 1.000 
Resultado esperado (1 a 5): p = 0.461 -> p ajustado = 1.000 
Acessibilidade considerada (1 a 5): p = 0.149 -> p ajustado = 0.894 
Recursos de acessibilidade (0 a 8): p = 0.310 -> p ajustado = 1.000 
Tipos de teste realizados: p = 0.457 -> p ajustado = 1.000 
Relatou alguma dificuldade: p = 0.001 -> p ajustado = 0.011 (significativo)
